# Firn Change and Mass Balance

This notebook reads and visualises the firn tracking output produced by `firn_change.pro` (IDL) and the GLAMOS mass balance data for the six study glaciers. It also derives the borehole thermal regime classification used for the maps in Part 1.

**Firn reconstruction:** A distributed mass balance model (Huss et al., 2021) reconstructs gridded annual mass balance from 1970 to 2025 at 10 m resolution, driven by field measurements and constrained by observed long-term ice volume changes. Firn cover is tracked cumulatively at each grid cell — accumulation adds layers, ablation removes the most recent ones first. Cells continuously firn-covered for at least two years are classified as firn (to exclude single anomalous snow years); firn persisting for more than 20 years is reclassified as ice. This yields annual firn thickness and extent maps from 1970 to 2025, from which the time since firn loss is computed per grid cell.

**Part 1 – Firn change analysis**
- Borehole thermal regime classification (used for map overlays)
- Load IDL-generated firn thickness, firn age, and time-since-firn-loss grids
- Map current firn thickness and age (2025)
- Map the time since firn loss per cell — the key predictor of englacial cooling
- Temporal evolution of firn extent (1980–2025)
- Statistics: annual firn area timeseries per glacier

**Part 2 – Mass balance figures**
- GLAMOS winter balance map mosaics
- Annual firn area timeseries (1970–2025)
- Spatially distributed annual mass balance maps (2024/25)

---

## Part 1 — Firn Change Analysis

## Imports


In [ ]:
%matplotlib inline
import matplotlib as mpl
mpl.rcParams['figure.dpi']   = 200
mpl.rcParams['font.family']  = 'Arial'

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import BoundaryNorm, TwoSlopeNorm
import geopandas as gpd
from glob import glob
from pathlib import Path
from affine import Affine

import pandas as pd
import matplotlib.ticker as mticker
import matplotlib.image as mpimg
from matplotlib.colors import ListedColormap
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection, Point, box as shapely_box
from shapely.prepared import prep

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.thermistor_plotting import build_profile_color_map
import src.gpr_processing as gpr
import src.gpr_plotting as gprp
from src.plot_composer import *
from src.geodata_processing import *
import cmcrameri.cm as cmc

### Paths and configuration

In [ ]:
# --- Input / output directories ---
firn_dir   = project_root + '/results/firn_grids/'
ortho_dir  = project_root + '/products/figures/gpr_figures/ice_thickness_maps/'
dems_dir   = ortho_dir + 'dems/'
output_dir = project_root + '/products/figures/firn_figures/'
os.makedirs(output_dir, exist_ok=True)

phd_root   = str(Path(project_root).parent.parent)   # .../PhD/
xyzn_dir   = project_root + '/data/raw/sgi_2022/xyzn_lv95/'

bh_csv = os.path.join(project_root, "data", "borehole_settings", "thermistor_coordinates.csv")

for label, path in [('bh_csv', bh_csv), ('xyzn_dir', xyzn_dir)]:
    print(f"{label}: {'OK' if os.path.exists(path) else 'NOT FOUND'}  ({path})")

# --- Glacier configuration ---
# force_ortho_download=True because old orthophotos cover only the small GPR
# survey extent; they must be re-downloaded for the full glacier outline bbox.
glaciers = [
    {'key': 'alphubel',  'dem_key': 'alphubel',  'ortho_key': 'alphubel',
     'label': 'Alphubel',  'abbr': 'AH', 'force_ortho_download': True,
     'xyzn_file': 'SGI_2023_B55-15_lv95.xyzn',
     'bh_ids': ['AH1G', 'AH2G', 'AH3G', 'AH1TT', 'AH2TT', 'AH3TT']},
    {'key': 'felskinn',  'dem_key': 'chessjen',  'ortho_key': 'chessjen',
     'label': 'Chessjen',  'abbr': 'CJ', 'force_ortho_download': True,
     'xyzn_file': 'SGI_2023_B53-14_lv95.xyzn',
     'bh_ids': ['CJ1G', 'CJ2G', 'CJ1TT', 'CJ2TT', 'CJ3TT', 'CJ4TT']},
    {'key': 'hohsaas',   'dem_key': 'hohsaas',   'ortho_key': 'hohsaas',
     'label': 'Hohsaas',   'abbr': 'HS', 'force_ortho_download': True,
     'xyzn_file': 'SGI_2023_B51-13_lv95.xyzn',
     'bh_ids': ['HS1G', 'HS2G', 'HS3G', 'HS1TT', 'HS2TT', 'HS3TT']},
    {'key': 'sexrouge',  'dem_key': 'sex_rouge',  'ortho_key': 'sexrouge',
     'label': 'Sex Rouge', 'abbr': 'SR', 'force_ortho_download': True,
     'xyzn_file': 'SGI_2023_B16-01_lv95.xyzn',
     'bh_ids': ['SR1TT', 'SR2TT']},
    {'key': 'tortin',    'dem_key': 'tortin',     'ortho_key': 'tortin',
     'label': 'Tortin',    'abbr': 'GT', 'force_ortho_download': True,
     'xyzn_file': 'SGI_2023_B75-12_lv95.xyzn',
     'bh_ids': ['GT1TT', 'GT2TT']},
    {'key': 'corvatsch', 'dem_key': 'corvatsch',  'ortho_key': 'corvatsch',
     'label': 'Corvatsch', 'abbr': 'CV', 'force_ortho_download': True,
     'xyzn_file': 'SGI_2022_E23-18_lv95.xyzn',
     'bh_ids': ['CV1TT', 'CV2TT']},
]

yrout         = [1980, 1990, 2000, 2010, 2014, 2019, 2022, 2024, 2025]
ANNO_FONTSIZE = 12
ABBR_FONTSIZE = 12
MAP_BUFFER_M  = 200   # buffer (m) around glacier extent when no outline found

## Borehole Thermal Regime Classification

Boreholes are classified by their thermal regime based on the fraction of sensor measurements at or above the pressure melting point. This classification is used as an overlay in the firn and time-since-firn-loss maps below.

In [ ]:
# ── Borehole thermal regime classification ────────────────────────────────────
# Temperate fraction = share of all sensor×hourly measurements where T ≥ −ε
#   ε = 0.05 °C (Geoprecision) / 0.20 °C (Tynitag)
#
# Categories:
#   cold-based    : to_bed=True AND frac_temperate < 0.02
#                   (entire ice column frozen all the way to the bed)
#   mostly cold   : frac_temperate < 0.5  (and not cold-based)
#   mostly temperate: frac_temperate ≥ 0.5

_sub_dir = sorted(Path(project_root).glob('results/glenglat_submission/*/'))[-1]
bh_sub  = pd.read_csv(_sub_dir / 'borehole.csv')
meas    = pd.read_csv(_sub_dir / 'measurement.csv')

EPS_GP = 0.05   # °C, Geoprecision precision
EPS_TT = 0.20   # °C, Tynitag precision

rows_cls = []
for _, bh in bh_sub.iterrows():
    m = meas[meas['borehole_id'] == bh['id']]
    if m.empty:
        continue
    eps = EPS_GP if bh['label'].endswith('G') else EPS_TT
    frac_temperate = (m['temperature'] >= -eps).mean()
    cold_based = (str(bh['to_bed']).lower() == 'true') and (frac_temperate < 0.02)

    if frac_temperate >= 0.5:
        regime = 'mostly temperate'
    elif cold_based:
        regime = 'cold-based'
    else:
        regime = 'mostly cold'

    rows_cls.append({
        'label': bh['label'], 'to_bed': bh['to_bed'],
        'frac_temperate': round(frac_temperate, 3),
        'regime': regime,
    })

df_cls = pd.DataFrame(rows_cls).set_index('label')
display(df_cls)

# Manual overrides (site-specific knowledge)
MANUAL_REGIME = {'CJ2G': 'cold-based'}
for _lbl, _reg in MANUAL_REGIME.items():
    if _lbl in df_cls.index:
        df_cls.loc[_lbl, 'regime'] = _reg

REGIME_COLORS = {
    'cold-based':       '#2166ac',   # dark blue
    'mostly cold':      '#92c5de',   # light blue
    'mostly temperate': '#d6604d',   # orange-red
}

def bh_color(name):
    return REGIME_COLORS.get(df_cls.loc[name, 'regime'] if name in df_cls.index else '', '#888888')

print('\nColor legend:', REGIME_COLORS)


### Helper: read Arc ASCII grid

Parses the 6-line header and returns a masked numpy array, an affine transform (LV95 / EPSG:2056), and the bounding box.

In [ ]:
def read_arc_grid(path):
    """Read an Arc ASCII (.grid) file.

    Handles:
    - IDL-style line wrapping (rows may span multiple file lines)
    - LV03 → LV95 coordinate conversion (offset +2e6 / +1e6) when
      the grid is detected to be in the old Swiss coordinate system
    - Tight bounding box derived from valid (non-NaN) cells only

    Returns
    -------
    data : np.ndarray (2-D, float32)
        Grid values; nodata cells set to np.nan.
    transform : affine.Affine
        Georeferencing transform (LV95).
    bbox : tuple
        Tight (xmin, ymin, xmax, ymax) around valid cells, in LV95.
    """
    with open(path) as f:
        ncols     = int(  f.readline().split()[1])
        nrows     = int(  f.readline().split()[1])
        xllcorner = float(f.readline().split()[1])
        yllcorner = float(f.readline().split()[1])
        cellsize  = float(f.readline().split()[1])
        nodata    = float(f.readline().split()[1])
        data = np.array(f.read().split(), dtype=np.float32).reshape(nrows, ncols)
    data[data == nodata] = np.nan

    # LV03 → LV95: Easting < 1e6 means old coordinate system
    if xllcorner < 1_000_000:
        xllcorner += 2_000_000
        yllcorner += 1_000_000

    # Arc ASCII row 0 = northernmost row  →  negative y pixel size
    transform = Affine(cellsize, 0, xllcorner,
                       0, -cellsize, yllcorner + nrows * cellsize)

    # Tight bbox from valid (non-NaN) cells, ignoring empty model margins
    valid = ~np.isnan(data)
    valid_rows = np.where(np.any(valid, axis=1))[0]
    valid_cols = np.where(np.any(valid, axis=0))[0]
    if len(valid_rows) and len(valid_cols):
        north = yllcorner + nrows * cellsize
        xmin = xllcorner + valid_cols[0]  * cellsize
        xmax = xllcorner + (valid_cols[-1] + 1) * cellsize
        ymax = north     - valid_rows[0]  * cellsize
        ymin = north     - (valid_rows[-1] + 1) * cellsize
    else:
        xmin = xllcorner
        xmax = xllcorner + ncols * cellsize
        ymin = yllcorner
        ymax = yllcorner + nrows * cellsize

    bbox = (xmin, ymin, xmax, ymax)
    return data, transform, bbox

### Load firn result grids

In [ ]:
for g in glaciers:
    k = g['key']

    # --- static outputs ---
    g['firnthick'],       g['tfm'], g['bbox'] = read_arc_grid(firn_dir + f'current/firnthick_{k}.grid')
    g['firnage'],         _,        _         = read_arc_grid(firn_dir + f'current/firnage_{k}.grid')
    g['time_since_firn'], _,        _         = read_arc_grid(firn_dir + f'current/time_since_firn_{k}.grid')

    # --- temporal snapshots ---
    g['snapshots'] = {}
    for yr in yrout:
        path = firn_dir + f'snapshots/firn{yr}_{k}.grid'
        if os.path.exists(path):
            g['snapshots'][yr], _, _ = read_arc_grid(path)

    # bbox with buffer for nicer map display
    xmin, ymin, xmax, ymax = g['bbox']
    g['bbox_map'] = (xmin - MAP_BUFFER_M, ymin - MAP_BUFFER_M,
                     xmax + MAP_BUFFER_M, ymax + MAP_BUFFER_M)

    print(f"{g['label']:12s}  grid {g['firnthick'].shape}  "
          f"firn cover: {np.sum(g['firnthick'] > 0) / np.sum(~np.isnan(g['firnthick'])) * 100:.1f}%  "
          f"snapshots: {list(g['snapshots'].keys())}")

### Load map resources

Orthophotos and SwissALTI3D DEM tiles are reused from the ice-thickness map workflow where they exist; missing ones are downloaded from Swisstopo.

In [ ]:
def make_square_bbox(xmin, ymin, xmax, ymax, buffer=0):
    cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
    half = max(xmax - xmin, ymax - ymin) / 2 + buffer
    return (cx - half, cy - half, cx + half, cy + half)

BH_BUFFER_M      = 300   # buffer around borehole cluster → matches study_sites extents
OUTLINE_BUFFER_M = 500   # fallback buffer when no boreholes available

for g in glaciers:
    k  = g['key']
    dk = g['dem_key']
    ok = g['ortho_key']

    # --- 1. boreholes (loaded first so they can drive the bbox) ---
    if g['bh_ids'] and os.path.exists(bh_csv):
        bh, _ = gpr.load_borehole_positions(bh_csv, keep_names=g['bh_ids'])
    else:
        bh = gpd.GeoDataFrame(columns=['geometry'], geometry='geometry', crs='EPSG:2056')
    g['boreholes'] = bh

    # --- 2. bbox_map: driven by borehole cluster if available, else outline ---
    if len(bh) > 0:
        g['bbox_map'] = tuple(gpr.square_bbox_from_gdf(bh, buffer_m=BH_BUFFER_M, pixel_size=0.5))
    else:
        xyzn_path = os.path.join(xyzn_dir, g['xyzn_file'])
        if os.path.exists(xyzn_path):
            outline_tmp = read_xyzn_to_gdf(xyzn_path)
            g['bbox_map'] = make_square_bbox(*outline_tmp.total_bounds, OUTLINE_BUFFER_M)
        else:
            g['bbox_map'] = make_square_bbox(*g['bbox'], OUTLINE_BUFFER_M)

    # --- 3. glacier outline (xyzn) ---
    xyzn_path = os.path.join(xyzn_dir, g['xyzn_file'])
    if os.path.exists(xyzn_path):
        g['outline'] = read_xyzn_to_gdf(xyzn_path)
    else:
        print(f"  WARNING: xyzn not found for {g['label']}")
        g['outline'] = None

    # --- 4. orthophoto ---
    ortho_path = os.path.join(ortho_dir, f'{ok}_orthophoto.tif')
    if not os.path.exists(ortho_path) or g.get('force_ortho_download', False):
        print(f"Downloading orthophoto for {g['label']} ...")
        gpr.download_swisstopo_orthophoto(
            g['bbox_map'], ortho_path, crs_epsg=2056, pixel_size=0.5,
            layer='ch.swisstopo.swissimage', fmt='image/jpeg'
        )
    g['ortho_path'] = ortho_path

    # --- 5. DEM tiles ---
    dem_glob = sorted(glob(os.path.join(dems_dir, dk, '*.tif')))
    g['dem_tiles'] = dem_glob
    if not dem_glob:
        print(f"  WARNING: no DEM tiles for {g['label']} at {dems_dir}{dk}/")

    w = g['bbox_map'][2] - g['bbox_map'][0]
    print(f"{g['label']:12s}  square {int(w)}m  "
          f"outline: {'ok' if g['outline'] is not None else 'MISSING'}  "
          f"dem: {len(dem_glob)}  bh: {len(bh)}")

empty_gdf = gpd.GeoDataFrame(
    {'profile': [], 'geometry': gpd.GeoSeries([], crs='EPSG:2056')},
    geometry='geometry', crs='EPSG:2056'
)

---
## Map 1 — Current firn thickness (2025)

Cumulative firn water equivalent (m w.e.) per 10 m cell as of 2025.
Cells with zero firn are transparent; borehole positions are overlaid.

In [ ]:
cmap_firn    = plt.cm.Blues
vmax_thick   = 50.0
levels_thick = np.arange(0, vmax_thick + 1, 5.0)
norm_thick   = BoundaryNorm(levels_thick, 256, clip=True)

SP = dict(left=0.11, right=0.99, top=0.97, bottom=0.14, wspace=0.05, hspace=0.08)
fmt = mticker.FuncFormatter(lambda v, _: f"{int(v):,}".replace(',', "'"))

fig, axs = plt.subplots(2, 3, figsize=(11, 7), dpi=200)
fig.subplots_adjust(**SP)

for i, (g, ax) in enumerate(zip(glaciers, axs.flat)):
    draw_glacier_map(
        ax=ax,
        ortho_path=g['ortho_path'],
        bbox=g['bbox_map'],
        gdf_pts=empty_gdf,
        dem_tiles=g['dem_tiles'],
        boreholes=empty_gdf,
        title=g['label'],
        ANNO_FONTSIZE=ANNO_FONTSIZE,
        ABBR_FONTSIZE=ABBR_FONTSIZE,
        background='hillshade' if g['dem_tiles'] else 'ortho',
        outlines=g['outline'],
        show_borehole_labels=False,
        show_contours=False,
        ylabel=False,
        show_flow_arrow=False,
        panel=chr(ord('a') + i),
    )
    masked = np.where(g['firnthick'] > 0, g['firnthick'], np.nan)
    gprp.imshow_grid(ax, masked, g['tfm'], cmap=cmap_firn, alpha=0.85, norm=norm_thick, zorder=10)

    if len(g['boreholes']) > 0:
        bh = g['boreholes']
        for _, row in bh.iterrows():
            color = bh_color(row['name'])
            ax.scatter(row.geometry.x, row.geometry.y,
                       s=70, marker='o', facecolor=color, edgecolor='black', linewidth=1.2, zorder=21)

    add_panel_outline(ax, g['bbox_map'], color='black', linewidth=1.4)

    ax.set_xlim(g['bbox_map'][0], g['bbox_map'][2])
    ax.set_ylim(g['bbox_map'][1], g['bbox_map'][3])
    ax.set_aspect('equal', adjustable='box')

    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.xaxis.set_major_formatter(fmt)
    ax.tick_params(axis='x', rotation=0, labelsize=8)
    if i // 3 == 1:
        ax.set_xlabel("Easting (LV95) [m]", fontsize=ANNO_FONTSIZE)
    else:
        ax.set_xlabel('')

    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.yaxis.set_major_formatter(fmt)
    ax.tick_params(axis='y', rotation=90, labelsize=8)
    if i % 3 == 0:
        ax.set_ylabel("Northing (LV95) [m]", fontsize=ANNO_FONTSIZE)

cax_w = 0.50
cax_left = (SP['left'] + SP['right']) / 2 - cax_w / 2 - 0.08
cax_y, cax_h = 0.02, 0.022
cax = fig.add_axes([cax_left, cax_y, cax_w, cax_h])
cb  = fig.colorbar(plt.cm.ScalarMappable(norm=norm_thick, cmap=cmap_firn),
                   cax=cax, orientation='horizontal')
cb.set_label('Firn thickness 2025 [m w.e.]', fontsize=ANNO_FONTSIZE)
cb.ax.tick_params(labelsize=ANNO_FONTSIZE - 2)
from matplotlib.lines import Line2D as _Line2D
_leg_handles = [
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#2166ac',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Cold'),
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#92c5de',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Polythermal'),
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#d6604d',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Temperate'),
]
fig.legend(handles=_leg_handles, loc='center left',
           bbox_to_anchor=(cax_left + cax_w + 0.022, cax_y - 0.01),
           fontsize=ANNO_FONTSIZE - 2, frameon=False, ncol=1,
           handlelength=1.4, handleheight=1.2, borderpad=0.3,
           title='Borehole thermal structure', title_fontsize=ANNO_FONTSIZE,
           alignment='left')
plt.savefig(output_dir + 'firnthick_2025.pdf', dpi=150, bbox_inches='tight')
plt.show()

---
## Map 2 — Firn age (2025)

Mean age (years before 2025) of surviving firn layers at each cell.
Only firn-covered cells are shown.

In [ ]:
cmap_age = cmc.devon_r
vmax_age = 20.0
levels_age = np.arange(0, vmax_age + 1, 1.0)
norm_age   = BoundaryNorm(levels_age, 256, clip=True)

SP = dict(left=0.11, right=0.99, top=0.97, bottom=0.14, wspace=0.05, hspace=0.08)

fmt = mticker.FuncFormatter(lambda v, _: f"{int(v):,}".replace(',', "'"))

fig, axs = plt.subplots(2, 3, figsize=(11, 7), dpi=200)
fig.subplots_adjust(**SP)

for i, (g, ax) in enumerate(zip(glaciers, axs.flat)):
    draw_glacier_map(
        ax=ax, ortho_path=g['ortho_path'], bbox=g['bbox_map'],
        gdf_pts=empty_gdf, dem_tiles=g['dem_tiles'], boreholes=empty_gdf,
        title=g['label'], ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE,
        background='hillshade' if g['dem_tiles'] else 'ortho',
        outlines=g['outline'], show_borehole_labels=False, show_contours=False,
        ylabel=False, show_flow_arrow=False,
        panel=chr(ord('a') + i),
    )
    masked = np.where(g['firnage'] >= 0, g['firnage'], np.nan)
    gprp.imshow_grid(ax, masked, g['tfm'], cmap=cmap_age, alpha=0.85, norm=norm_age, zorder=10)

    if len(g['boreholes']) > 0:
        bh = g['boreholes']
        for _, row in bh.iterrows():
            color = bh_color(row['name'])
            ax.scatter(row.geometry.x, row.geometry.y,
                       s=70, marker='o', facecolor=color, edgecolor='black', linewidth=1.2, zorder=21)

    add_panel_outline(ax, g['bbox_map'], color='black', linewidth=1.4)

    ax.set_xlim(g['bbox_map'][0], g['bbox_map'][2])
    ax.set_ylim(g['bbox_map'][1], g['bbox_map'][3])
    ax.set_aspect('equal', adjustable='box')

    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.xaxis.set_major_formatter(fmt)
    ax.tick_params(axis='x', rotation=0, labelsize=8)
    if i // 3 == 1:
        ax.set_xlabel("Easting (LV95) [m]", fontsize=ANNO_FONTSIZE)
    else:
        ax.set_xlabel('')

    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.yaxis.set_major_formatter(fmt)
    ax.tick_params(axis='y', rotation=90, labelsize=8)
    if i % 3 == 0:
        ax.set_ylabel("Northing (LV95) [m]", fontsize=ANNO_FONTSIZE)

cax_w = 0.50
cax_left = (SP['left'] + SP['right']) / 2 - cax_w / 2 - 0.08
cax_y, cax_h = 0.02, 0.022
cax = fig.add_axes([cax_left, cax_y, cax_w, cax_h])
cb  = fig.colorbar(plt.cm.ScalarMappable(norm=norm_age, cmap=cmap_age),
                   cax=cax, orientation='horizontal')
cb.set_label('Mean firn age [yr before 2025]', fontsize=ANNO_FONTSIZE)
cb.ax.tick_params(labelsize=ANNO_FONTSIZE - 2)
from matplotlib.lines import Line2D as _Line2D
_leg_handles = [
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#2166ac',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Cold'),
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#92c5de',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Polythermal'),
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#d6604d',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Temperate'),
]
fig.legend(handles=_leg_handles, loc='center left',
           bbox_to_anchor=(cax_left + cax_w + 0.022, cax_y - 0.01),
           fontsize=ANNO_FONTSIZE - 2, frameon=False, ncol=1,
           handlelength=1.4, handleheight=1.2, borderpad=0.3,
           title='Borehole thermal structure', title_fontsize=ANNO_FONTSIZE,
           alignment='left')
plt.savefig(output_dir + 'firnage_2025.pdf', dpi=150, bbox_inches='tight')
plt.show()

---
## Map 3 — Time since firn loss

How many years ago firn disappeared at each cell (only for cells where firn lasted ≥ 2 consecutive years).
This is the variable most directly relevant for explaining cold ice temperatures —
cells where firn disappeared decades ago are expected to be coldest.

In [ ]:
# Blue bin prepended at left of 0 — same width as one red bin (2-yr spacing)
FIRN_BLUE     = '#4393c3'
FIRN_SENTINEL = -2.0

all_tsf    = np.concatenate([g['time_since_firn'][g['time_since_firn'] >= 0] for g in glaciers])
vmax_tsf   = np.percentile(all_tsf, 98) if len(all_tsf) else 50.0
red_levels = np.arange(0, vmax_tsf + 1, 2.0)
levels_all = np.concatenate([[FIRN_SENTINEL], red_levels])
n_red      = len(red_levels) - 1

blue_c    = np.array([mcolors.to_rgba(FIRN_BLUE, alpha=0.65)])
red_c     = cmc.lajolla_r(np.linspace(0, 1, n_red))
cmap_all  = mcolors.ListedColormap(np.vstack([blue_c, red_c]))
norm_all  = BoundaryNorm(levels_all, cmap_all.N)

SP  = dict(left=0.11, right=0.99, top=0.97, bottom=0.14, wspace=0.05, hspace=0.08)
fmt = mticker.FuncFormatter(lambda v, _: f"{int(v):,}".replace(',', "\'"))

fig, axs = plt.subplots(2, 3, figsize=(11, 7), dpi=200)
fig.subplots_adjust(**SP)

for i, (g, ax) in enumerate(zip(glaciers, axs.flat)):
    draw_glacier_map(
        ax=ax, ortho_path=g['ortho_path'], bbox=g['bbox_map'],
        gdf_pts=empty_gdf, dem_tiles=g['dem_tiles'], boreholes=empty_gdf,
        title=g['label'], ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE,
        background='hillshade' if g['dem_tiles'] else 'ortho',
        outlines=g['outline'], show_borehole_labels=False, show_contours=False,
        ylabel=False, show_flow_arrow=False,
        panel=chr(ord('a') + i),
    )

    combined = np.where(g['firnthick'] > 0, FIRN_SENTINEL,
                        np.where(g['time_since_firn'] >= 0, g['time_since_firn'], np.nan))
    gprp.imshow_grid(ax, combined, g['tfm'], cmap=cmap_all, alpha=0.85, norm=norm_all, zorder=10)

    if len(g['boreholes']) > 0:
        bh = g['boreholes']
        for _, row in bh.iterrows():
            color = bh_color(row['name'])
            ax.scatter(row.geometry.x, row.geometry.y,
                       s=70, marker='o', facecolor=color, edgecolor='black', linewidth=1.2, zorder=21)

    add_panel_outline(ax, g['bbox_map'], color='black', linewidth=1.4)

    ax.set_xlim(g['bbox_map'][0], g['bbox_map'][2])
    ax.set_ylim(g['bbox_map'][1], g['bbox_map'][3])
    ax.set_aspect('equal', adjustable='box')

    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.xaxis.set_major_formatter(fmt)
    ax.tick_params(axis='x', rotation=0, labelsize=8)
    if i // 3 == 1:
        ax.set_xlabel("Easting (LV95) [m]", fontsize=ANNO_FONTSIZE)
    else:
        ax.set_xlabel('')

    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.yaxis.set_major_formatter(fmt)
    ax.tick_params(axis='y', rotation=90, labelsize=8)
    if i % 3 == 0:
        ax.set_ylabel("Northing (LV95) [m]", fontsize=ANNO_FONTSIZE)

cax_w    = 0.50
cax_left = (SP['left'] + SP['right']) / 2 - cax_w / 2 - 0.08
cax_y    = 0.02
cax_h    = 0.022

cax = fig.add_axes([cax_left, cax_y, cax_w, cax_h])
cb  = fig.colorbar(plt.cm.ScalarMappable(norm=norm_all, cmap=cmap_all),
                   cax=cax, orientation='horizontal')

red_ticks      = np.arange(0, vmax_tsf + 1, 6)
all_ticks      = np.concatenate([[FIRN_SENTINEL / 2], red_ticks])
all_labels_out = [''] + [str(int(v)) for v in red_ticks]
cb.set_ticks(all_ticks)
cb.set_ticklabels(all_labels_out)
cb.ax.tick_params(labelsize=ANNO_FONTSIZE - 2)
cb.ax.axvline(x=0, color='black', linewidth=0.8, zorder=5)

x_frac = float((FIRN_SENTINEL / 2 - levels_all[0]) / (levels_all[-1] - levels_all[0]))
cb.ax.annotate('Firn\ncovered',
               xy=(x_frac, 0), xytext=(x_frac, -1.2),
               xycoords='axes fraction', textcoords='axes fraction',
               ha='center', va='top', fontsize=ANNO_FONTSIZE - 2,
               annotation_clip=False,
               arrowprops=dict(arrowstyle='-', color='black', lw=0.8))
cb.set_label('Years since firn loss', fontsize=ANNO_FONTSIZE, labelpad=2)

from matplotlib.lines import Line2D as _Line2D
_leg_handles = [
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#2166ac',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Cold'),
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#92c5de',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Polythermal'),
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#d6604d',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Temperate'),
]
_bh_legend = fig.legend(handles=_leg_handles, loc='center left',
           bbox_to_anchor=(cax_left + cax_w + 0.022, cax_y - 0.01),
           fontsize=ANNO_FONTSIZE - 2, frameon=False, ncol=1,
           handlelength=1.4, handleheight=1.2, borderpad=0.3,
           title='Borehole thermal structure', title_fontsize=ANNO_FONTSIZE,
           alignment='left')
plt.savefig(os.path.join(project_root, "figures", "paper", "fig08_time_since_firn.pdf"), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Per-borehole years since firn loss

rows = []
for g in glaciers:
    tsf  = g['time_since_firn']
    tfm  = g['tfm']
    nrows, ncols = tsf.shape
    for _, bh in g['boreholes'].iterrows():
        x, y = bh.geometry.x, bh.geometry.y
        col = int((x - tfm.c) / tfm.a)
        row = int((y - tfm.f) / tfm.e)
        if 0 <= row < nrows and 0 <= col < ncols:
            val = tsf[row, col]
            if np.isnan(val) or val < 0:
                yrs = "no data"
            elif val == 0:
                yrs = "firn present"
            else:
                yrs = f"{int(round(val))} yrs ago"
        else:
            yrs = "outside grid"
        rows.append({'Glacier': g['label'], 'Borehole': bh['name'],
                     'Years since firn loss': yrs})

df_tsf = pd.DataFrame(rows)
display(df_tsf)

---
## Map 4 — Temporal evolution of firn thickness

One row per glacier, one column per snapshot year.
Shows how the firn extent has changed from 1980 to 2025.
Adjust `plot_years` to select a subset of `yrout`.

In [ ]:
plot_years = [1980, 2000, 2010, 2019, 2022, 2025]  # subset of yrout

ncols = len(plot_years)
nrows = len(glaciers)

cmap_snap = plt.cm.Blues

# vmax from actual data (98th percentile across all snapshots)
all_snap = np.concatenate([
    g['snapshots'][yr][~np.isnan(g['snapshots'][yr]) & (g['snapshots'][yr] > 0)]
    for g in glaciers for yr in plot_years if yr in g['snapshots']
])
vmax_snap   = np.percentile(all_snap, 98) if len(all_snap) else 50.0
levels_snap = np.arange(0, vmax_snap + 1, 2.0)
norm_snap   = BoundaryNorm(levels_snap, 256, clip=True)

SP = dict(left=0.07, right=0.99, top=0.97, bottom=0.08, wspace=0.05, hspace=0.05)

# Darken the hillshade background for this figure only (scoped to this cell) so the
# brightest sunlit terrain doesn't wash out against the near-white low-thickness firn overlay
draw_glacier_map._hillshade_defaults = {'vmax': 1.15}

fig, axs = plt.subplots(nrows, ncols,
                        figsize=(2.5 * ncols, 3.0 * nrows), dpi=200)
fig.subplots_adjust(**SP)

for r, g in enumerate(glaciers):
    for c, yr in enumerate(plot_years):
        ax = axs[r, c]

        draw_glacier_map(
            ax=ax,
            ortho_path=g['ortho_path'],
            bbox=g['bbox_map'],
            gdf_pts=empty_gdf,
            dem_tiles=g['dem_tiles'],
            boreholes=empty_gdf,
            title='',
            ANNO_FONTSIZE=ANNO_FONTSIZE - 2,
            ABBR_FONTSIZE=ABBR_FONTSIZE - 2,
            background='hillshade',
            outlines=g['outline'],
            show_borehole_labels=False,
            show_contours=False,
            ylabel=False,
            show_flow_arrow=False,
        )

        if yr in g['snapshots']:
            masked = np.where(g['snapshots'][yr] > 0, g['snapshots'][yr], np.nan)
            gprp.imshow_grid(ax, masked, g['tfm'], cmap=cmap_snap,
                             alpha=0.85, norm=norm_snap, zorder=10)

        # Boreholes only in 2025 panel — not drilled in earlier years
        if yr == 2025 and len(g['boreholes']) > 0:
            bh = g['boreholes']
            for _, row in bh.iterrows():
                color = bh_color(row['name'])
                ax.scatter(row.geometry.x, row.geometry.y,
                           s=40, marker='o', facecolor=color, edgecolor='black',
                           linewidth=1.0, zorder=21)

        add_panel_outline(ax, g['bbox_map'], color='black', linewidth=1.0)
        ax.set_xlim(g['bbox_map'][0], g['bbox_map'][2])
        ax.set_ylim(g['bbox_map'][1], g['bbox_map'][3])
        ax.set_aspect('equal', adjustable='box')

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')

        # Glacier label on left column, year label on top row
        if c == 0:
            ax.text(0.03, 0.97, g['label'], transform=ax.transAxes,
                    fontsize=ANNO_FONTSIZE - 1, fontweight='bold',
                    va='top', ha='left', zorder=30,
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.8))
        if r == 0:
            ax.set_title(str(yr), fontsize=ANNO_FONTSIZE + 2, fontweight='bold', pad=4)

draw_glacier_map._hillshade_defaults = None  # reset so later figures aren't affected

# Shared colorbar centred over subplot area
cax_w = 0.50
cax_left = (SP['left'] + SP['right']) / 2 - cax_w / 2 - 0.08
cax_y, cax_h = 0.03, 0.018
cax = fig.add_axes([cax_left, cax_y, cax_w, cax_h])
cb  = fig.colorbar(plt.cm.ScalarMappable(norm=norm_snap, cmap=cmap_snap),
                   cax=cax, orientation='horizontal')
cb.set_label('Firn thickness [m w.e.]', fontsize=ANNO_FONTSIZE + 1)
cb.ax.tick_params(labelsize=ANNO_FONTSIZE)

from matplotlib.lines import Line2D as _Line2D
_leg_handles = [
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#2166ac',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Cold'),
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#92c5de',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Polythermal'),
    _Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#d6604d',
            markeredgecolor='black', markeredgewidth=0.8, markersize=8, label='Temperate'),
]
fig.legend(handles=_leg_handles, loc='center left',
           bbox_to_anchor=(cax_left + cax_w + 0.022, cax_y + cax_h / 2),
           fontsize=ANNO_FONTSIZE - 2, frameon=False, ncol=1,
           handlelength=1.4, handleheight=1.2, borderpad=0.3,
           title='Borehole thermal structure', title_fontsize=ANNO_FONTSIZE + 1,
           alignment='left')
plt.savefig(output_dir + 'firn_temporal_evolution.pdf', dpi=100, bbox_inches='tight')

plt.savefig(os.path.join(project_root, 'figures', 'supplement', 'figS10_firn_temporal_evolution.pdf'), dpi=100, bbox_inches='tight')
plt.show()

---
## Statistics — Firn loss and evolution per glacier

In [ ]:
CELL_AREA_KM2 = 10 * 10 * 1e-6   # 10 m cell → km²
SNAP_YEARS    = sorted(yrout)

# ── load annual firn area CSVs (written by IDL per glacier) ──────────────────
annual_ts = {}   # label → pd.Series(year → firn_area_km2)
for g in glaciers:
    csv_path = firn_dir + f'annual_area/firn_area_annual_{g["key"]}.csv'
    try:
        df_csv = pd.read_csv(csv_path)
        df_csv['firn_area_km2'] = df_csv['firn_cells'] * CELL_AREA_KM2
        annual_ts[g['label']] = pd.Series(
            df_csv['firn_area_km2'].values,
            index=df_csv['year'].values
        )
    except FileNotFoundError:
        annual_ts[g['label']] = None

# ── compute per-glacier statistics ──────────────────────────────────────────
records     = []
time_series = {}   # label → {year: firn_cover_pct}

for g in glaciers:
    thick = g['firnthick']
    age   = g['firnage']
    tsf   = g['time_since_firn']
    snaps = g['snapshots']

    glacier_area_km2 = np.sum(~np.isnan(thick)) * CELL_AREA_KM2

    # Use annual CSV if available, else fall back to snapshots
    ann = annual_ts[g['label']]
    if ann is not None:
        ts_km2 = {int(yr): float(v) for yr, v in ann.items()}
    else:
        ts_km2 = {}
        for yr in SNAP_YEARS:
            if yr in snaps:
                ts_km2[yr] = np.nansum(snaps[yr] > 0) * CELL_AREA_KM2

    # Convert to % of glacier area
    ts = {yr: v / glacier_area_km2 * 100 for yr, v in ts_km2.items()} if glacier_area_km2 > 0 else ts_km2
    time_series[g['label']] = ts

    f1980_km2  = ts_km2.get(1980, 0.0)
    f2025_km2  = ts_km2.get(2025, 0.0)
    f_max_km2  = max(ts_km2.values()) if ts_km2 else 0.0
    yr_max     = max(ts_km2, key=ts_km2.get) if ts_km2 else None

    years_with_firn = [yr for yr, a in ts_km2.items() if a > 0]
    last_yr = max(years_with_firn) if years_with_firn else None

    cov1980 = f1980_km2 / glacier_area_km2 * 100 if glacier_area_km2 > 0 else 0.0
    cov2025 = f2025_km2 / glacier_area_km2 * 100 if glacier_area_km2 > 0 else 0.0
    pct_change = (f2025_km2 - f1980_km2) / f1980_km2 * 100 if f1980_km2 > 0 else None

    thick_pos = thick[np.isfinite(thick) & (thick > 0)]
    mean_thick = float(np.mean(thick_pos)) if len(thick_pos) else None

    age_pos = age[np.isfinite(age) & (age >= 0)]
    mean_age = float(np.mean(age_pos)) if len(age_pos) else None

    tsf_pos = tsf[np.isfinite(tsf) & (tsf >= 0)]
    mean_tsf = float(np.mean(tsf_pos)) if len(tsf_pos) else None

    def _fmt(v, decimals=2):
        return f'{v:.{decimals}f}' if v is not None else '—'

    records.append({
        'Glacier':                       g['label'],
        'Area\n[km²]':                   f'{glacier_area_km2:.2f}',
        'Firn 1980\n[km²  /  %]':        f'{f1980_km2:.3f}  /  {cov1980:.0f}',
        'Firn 2025\n[km²  /  %]':        f'{f2025_km2:.3f}  /  {cov2025:.0f}',
        'Change\n1980→2025 [%]':         _fmt(pct_change, 0),
        'Peak firn\n[km² / yr]':         f'{f_max_km2:.3f}  /  {yr_max}' if yr_max else '—',
        'Mean thick.\n2025 [m w.e.]':    _fmt(mean_thick, 1),
        'Mean firn\nage [yr]':           _fmt(mean_age, 1),
        'Mean yrs since\nfirn loss':     _fmt(mean_tsf, 1),
    })

df = pd.DataFrame(records).set_index('Glacier')
print(df.to_string())

# ── colours via build_profile_color_map (romaO cyclic) ──────────────────────
glacier_labels = [g['label'] for g in glaciers]
COLORS = build_profile_color_map(glacier_labels)

# ── figure ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(11, 6), dpi=200)
gs  = fig.add_gridspec(2, 1, height_ratios=[1, 0.9], hspace=0.24)

# — time series ———————————————————————————————————————————————————————————————
ax = fig.add_subplot(gs[0])
for g in glaciers:
    ts   = time_series[g['label']]
    yrs  = sorted(ts.keys())
    vals = [ts[y] for y in yrs]
    has_annual = annual_ts[g['label']] is not None
    ax.plot(yrs, vals,
            '-' if has_annual else 'o-',
            label=g['label'],
            color=COLORS[g['label']], lw=1.8, ms=5, zorder=5)

ax.set_xlabel('Year', fontsize=ANNO_FONTSIZE)
ax.set_ylabel('Firn cover [% of 2023 glacier area]', fontsize=ANNO_FONTSIZE)
leg = ax.legend(fontsize=ANNO_FONTSIZE - 3, ncol=len(glaciers), framealpha=1.0,
                loc='lower center', bbox_to_anchor=(0.5, 1.06),
                edgecolor='black', fancybox=False, borderaxespad=0)
ax.tick_params(labelsize=ANNO_FONTSIZE - 2)
ax.set_xlim(1978, 2028)
ax.set_ylim(0, 105)
ax.xaxis.set_major_locator(plt.MultipleLocator(5))
ax.grid(True, alpha=0.25, linestyle='--')
ax.text(-0.09, 1.1, '(a)', transform=ax.transAxes,
        ha='left', va='top', fontsize=ABBR_FONTSIZE, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=1.0))

# — table —————————————————————————————————————————————————————————————————————
ax_tb = fig.add_subplot(gs[1])
ax_tb.axis('off')

tbl = ax_tb.table(
    cellText=df.values.tolist(),
    rowLabels=list(df.index),
    colLabels=list(df.columns),
    cellLoc='center',
    rowLoc='center',
    loc='upper center',
    bbox=[0, 0, 1, 1],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(ANNO_FONTSIZE - 3)
tbl.scale(1, 1.6)

HEADER_COLOR = '#4d4d4d'
ROW_COLOR    = '#f0f0f0'

for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor('black')
    if r == 0:
        cell.set_facecolor(HEADER_COLOR)
        cell.set_text_props(color='white', fontweight='bold')
    elif c == -1:
        cell.set_text_props(fontweight='bold')
        cell.set_facecolor(ROW_COLOR)
    else:
        cell.set_facecolor(ROW_COLOR)

ax_tb.text(-0.09, 1.01, '(b)', transform=ax_tb.transAxes,
           ha='left', va='bottom', fontsize=ABBR_FONTSIZE, fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=1.0))

plt.savefig(output_dir + 'firn_statistics.pdf', dpi=150, bbox_inches='tight')

plt.savefig(os.path.join(project_root, 'figures', 'supplement', 'figS11_firn_statistics.pdf'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
"""Compress this notebook's PDF figures to ≤ 2 MB using Ghostscript."""
import subprocess, shutil

MAX_BYTES = 2 * 1024 * 1024  # 2 MB
GS_BIN = shutil.which('gs') or '/usr/local/bin/gs'

fig_files = [
    Path(output_dir) / 'firnthick_2025.pdf',
    Path(output_dir) / 'firnage_2025.pdf',
    Path(project_root) / 'figures' / 'paper' / 'fig08_time_since_firn.pdf',
    Path(output_dir) / 'firn_temporal_evolution.pdf',
    Path(output_dir) / 'firn_statistics.pdf',
    Path(project_root) / 'figures' / 'supplement' / 'figS10_firn_temporal_evolution.pdf',
    Path(project_root) / 'figures' / 'supplement' / 'figS11_firn_statistics.pdf',
]

GS_SETTINGS = ['/printer', '/ebook', '/screen']  # tried in order, increasing compression

def compress_pdf(path: Path, max_bytes: int = MAX_BYTES) -> None:
    original_bytes = path.stat().st_size
    if original_bytes <= max_bytes:
        print(f"  ok   {path.name}  ({original_bytes/1e6:.2f} MB)")
        return
    tmp = path.with_suffix('.compressed.pdf')
    for setting in GS_SETTINGS:
        result = subprocess.run([
            GS_BIN, '-dBATCH', '-dNOPAUSE', '-q',
            '-sDEVICE=pdfwrite', '-dCompatibilityLevel=1.4',
            f'-dPDFSETTINGS={setting}',
            f'-sOutputFile={tmp}', str(path)
        ], capture_output=True)
        if result.returncode != 0:
            print(f"  ERROR running gs on {path.name}: {result.stderr.decode()}")
            tmp.unlink(missing_ok=True)
            return
        compressed_bytes = tmp.stat().st_size
        if compressed_bytes <= max_bytes:
            shutil.move(tmp, path)
            print(f"  compressed  {path.name}: "
                  f"{original_bytes/1e6:.2f} MB → {compressed_bytes/1e6:.2f} MB  "
                  f"(gs {setting})")
            return
        tmp.unlink(missing_ok=True)
    print(f"  WARNING  {path.name}: still {compressed_bytes/1e6:.2f} MB after all gs settings")

for p in fig_files:
    if p.exists():
        compress_pdf(p)
    else:
        print(f"  not found: {p.name}")
print("\nDone.")


---

## Part 2 — Mass Balance Figures

In [ ]:
output_dir = project_root + '/products/figures/mass_balance_figures/'
os.makedirs(output_dir, exist_ok=True)

---
## Figure 1 — Spatially distributed winter balance (2024/25)

Six-panel map showing the GLAMOS modelled spatially distributed winter balance
for the 2024/25 hydrological year at all six study glaciers.
Data source: `data/glamos/distributed_mass_balance_grids/` (LV95, 10 m resolution).

In [ ]:
import sys, os



MB_DIR    = project_root + '/data/glamos/distributed_mass_balance_grids/'
PT_WINTER = '/Volumes/jabeer/glazioarch/GlacioBaseData/MassBalance/point/winter/'
PT_ANNUAL = '/Volumes/jabeer/glazioarch/GlacioBaseData/MassBalance/point/annual/'
xyzn_dir  = project_root + '/data/raw/sgi_2022/xyzn_lv95/'
bh_csv    = os.path.join(project_root, "data", "borehole_settings", "thermistor_coordinates.csv")
output_dir = project_root + '/products/figures/mass_balance_figures/'
os.makedirs(output_dir, exist_ok=True)

ANNO_FONTSIZE = 12
MAP_BUFFER_M  = 100

glaciers_mb = [
    {'key': 'alphubel',  'label': 'Alphubel (AH)',       'abbr': 'AH',
     'mb_path':    MB_DIR + 'alphubel/2025_ann_fix.grid',
     'pt_key':     'alphubel',
     'xyzn_file':  'SGI_2023_B55-15_lv95.xyzn',
     'bh_ids':     ['AH1G', 'AH2G', 'AH3G', 'AH1TT', 'AH2TT', 'AH3TT'],
     'bh_filter':  True},
    {'key': 'chessjen',  'label': 'Chessjen (CJ)',    'abbr': 'CJ',
     'mb_path':    MB_DIR + 'chessjen/2025_ann_fix.grid',
     'pt_key':     'felskinn',
     'xyzn_file':  'SGI_2023_B53-14_lv95.xyzn',
     'bh_ids':     ['CJ1G', 'CJ2G', 'CJ1TT', 'CJ2TT', 'CJ3TT', 'CJ4TT'],
     'bh_filter':  False},
    {'key': 'hohsaas',   'label': 'Hohsaas (HS)',     'abbr': 'HS',
     'mb_path':    MB_DIR + 'hohsaas/2025_ann_fix.grid',
     'pt_key':     'hohsaas',
     'xyzn_file':  'SGI_2023_B51-13_lv95.xyzn',
     'bh_ids':     ['HS1G', 'HS2G', 'HS3G', 'HS1TT', 'HS2TT', 'HS3TT'],
     'bh_filter':  True},
    {'key': 'sex_rouge', 'label': 'Sex Rouge (SR)', 'abbr': 'SR',
     'mb_path':    MB_DIR + 'sex_rouge/2025_ann_fix.grid',
     'pt_key':     'sexrouge',
     'xyzn_file':  'SGI_2023_B16-01_lv95.xyzn',
     'bh_ids':     ['SR1TT', 'SR2TT'],
     'bh_filter':  True},
    {'key': 'tortin',    'label': 'Tortin (GT)',    'abbr': 'GT',
     'mb_path':    MB_DIR + 'tortin/2025_ann_fix.grid',
     'pt_key':     'tortin',
     'xyzn_file':  'SGI_2023_B75-12_lv95.xyzn',
     'bh_ids':     ['GT1TT', 'GT2TT'],
     'bh_filter':  True},
    {'key': 'corvatsch', 'label': 'Corvatsch (CV)', 'abbr': 'CV',
     'mb_path':    MB_DIR + 'corvatsch/2025_ann_fix.grid',
     'pt_key':     'corvatsch',
     'xyzn_file':  'SGI_2022_E23-18_lv95.xyzn',
     'bh_ids':     ['CV1TT', 'CV2TT'],
     'bh_filter':  True},
]

def read_arc_grid(path):
    with open(path) as f:
        ncols     = int(float(f.readline().split()[1]))
        nrows     = int(float(f.readline().split()[1]))
        xllcorner = float(f.readline().split()[1])
        yllcorner = float(f.readline().split()[1])
        cellsize  = float(f.readline().split()[1])
        nodata    = float(f.readline().split()[1])
        data = np.array(f.read().split(), dtype=np.float32).reshape(nrows, ncols)
    data[data == nodata] = np.nan
    if xllcorner < 1_000_000:
        xllcorner += 2_000_000
        yllcorner += 1_000_000
    transform = Affine(cellsize, 0, xllcorner,
                       0, -cellsize, yllcorner + nrows * cellsize)
    valid = ~np.isnan(data)
    valid_rows = np.where(np.any(valid, axis=1))[0]
    valid_cols = np.where(np.any(valid, axis=0))[0]
    north = yllcorner + nrows * cellsize
    xmin = xllcorner + valid_cols[0]  * cellsize
    xmax = xllcorner + (valid_cols[-1] + 1) * cellsize
    ymax = north     - valid_rows[0]  * cellsize
    ymin = north     - (valid_rows[-1] + 1) * cellsize
    return data, transform, (xmin, ymin, xmax, ymax)

def make_square_bbox(xmin, ymin, xmax, ymax, buffer=0):
    cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
    half = max(xmax - xmin, ymax - ymin) / 2 + buffer
    return (cx - half, cy - half, cx + half, cy + half)

def read_mb_points(filepath, year=2025):
    """Read GLAMOS point mass balance file (LV03 → LV95), filter for balance year ending `year`."""
    xs, ys = [], []
    if not os.path.exists(filepath):
        return np.array([]), np.array([])
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 10:
                continue
            if not parts[3].startswith(str(year)):
                continue
            try:
                x = float(parts[7])
                y = float(parts[8])
            except (ValueError, IndexError):
                continue
            if not (450_000 < x < 900_000 and 65_000 < y < 300_000):
                continue
            xs.append(x + 2_000_000)
            ys.append(y + 1_000_000)
    return np.array(xs), np.array(ys)

empty_gdf = gpd.GeoDataFrame(
    {'profile': [], 'geometry': gpd.GeoSeries([], crs='EPSG:2056')},
    geometry='geometry', crs='EPSG:2056')

for g in glaciers_mb:
    g['mb_data'], g['tfm'], g['bbox'] = read_arc_grid(g['mb_path'])
    g['bbox_map'] = make_square_bbox(*g['bbox'], MAP_BUFFER_M)
    g['winter_pts'] = read_mb_points(PT_WINTER + f"{g['pt_key']}_winter.dat")
    g['annual_pts'] = read_mb_points(PT_ANNUAL + f"{g['pt_key']}_annual.dat")
    xyzn_path = os.path.join(xyzn_dir, g['xyzn_file'])
    g['outline'] = read_xyzn_to_gdf(xyzn_path) if os.path.exists(xyzn_path) else None
    if g['bh_ids'] and os.path.exists(bh_csv):
        bh, _ = gpr.load_borehole_positions(bh_csv, keep_names=g['bh_ids'])
    else:
        bh = empty_gdf
    g['boreholes'] = bh
    valid = g['mb_data'][~np.isnan(g['mb_data'])]
    print(f"{g['label']:22s}  Ba={np.mean(valid):.2f} m w.e.  "
          f"winter_pts={len(g['winter_pts'][0]):2d}  annual_pts={len(g['annual_pts'][0]):2d}  "
          f"outline={'ok' if g['outline'] is not None else 'MISSING'}  bh={len(bh)}")


In [ ]:
levels_mb = np.arange(-4.0, 2.0 + 0.01, 0.5)
n_neg = int(np.sum(levels_mb < 0))   # bins below 0
n_pos = int(np.sum(levels_mb > 0))   # bins above 0
# Sample lower half of vik_r for negatives, upper half for positives,
# so that the colormap neutral falls exactly at the 0 boundary.
neg_colors = cmc.vik_r(np.linspace(0, 0.5, n_neg, endpoint=False))
pos_colors = cmc.vik_r(np.linspace(0.5, 1.0, n_pos + 1)[1:])
cmap_mb = ListedColormap(np.vstack([neg_colors, pos_colors]))
norm_mb   = BoundaryNorm(levels_mb, cmap_mb.N)

SP  = dict(left=0.11, right=0.99, top=0.97, bottom=0.14, wspace=0.05, hspace=0.08)
fmt = mticker.FuncFormatter(lambda v, _: f"{int(v):,}".replace(',', "'"))

fig, axs = plt.subplots(2, 3, figsize=(11, 7), dpi=200)
fig.subplots_adjust(**SP)

def draw_outline(ax, geoms_iter, color='black', lw=1.8, alpha=0.9, zorder=11):
    segs = []
    for geom in geoms_iter:
        if geom is None or geom.is_empty:
            continue
        parts_raw = list(geom.geoms) if isinstance(geom, (GeometryCollection, MultiPolygon)) else [geom]
        for part in parts_raw:
            if isinstance(part, Polygon) and not part.is_empty:
                segs.append(list(part.exterior.coords))
            elif isinstance(part, MultiPolygon):
                for p in part.geoms:
                    if not p.is_empty:
                        segs.append(list(p.exterior.coords))
    if segs:
        ax.add_collection(LineCollection(segs, colors=color, linewidths=lw,
                                         alpha=alpha, zorder=zorder))

def prepare_outline(gdf_raw, mb_bbox, panel_bbox, bh_points=None, bh_filter=True):
    mb_box  = shapely_box(*mb_bbox)
    pan_box = shapely_box(*panel_bbox)
    result  = []
    for geom in gdf_raw.geometry:
        if geom is None or geom.is_empty:
            continue
        if not geom.is_valid:
            try:
                geom = geom.buffer(0)
            except Exception:
                continue
        try:
            ovlp_area = geom.intersection(mb_box).area
        except Exception:
            ovlp_area = 0
        if ovlp_area < 5000:
            continue
        if bh_filter and bh_points:
            if not any(geom.contains(pt) for pt in bh_points):
                continue
        try:
            clipped = geom.intersection(pan_box)
        except Exception:
            clipped = geom
        if clipped is None or clipped.is_empty:
            continue
        if isinstance(clipped, (Polygon, MultiPolygon)) and clipped.area < 1.0:
            continue
        result.append(clipped)
    return result

def explode_outline_geoms(geoms):
    parts = []
    for geom in geoms:
        if geom is None or geom.is_empty:
            continue
        if isinstance(geom, MultiPolygon):
            parts.extend([part for part in geom.geoms if not part.is_empty])
        elif isinstance(geom, GeometryCollection):
            for part in geom.geoms:
                if isinstance(part, MultiPolygon):
                    parts.extend([subpart for subpart in part.geoms if not subpart.is_empty])
                elif isinstance(part, Polygon) and not part.is_empty:
                    parts.append(part)
        else:
            parts.append(geom)
    return parts

def keep_supported_outline_geoms(geoms, mb_data, transform):
    valid_rows, valid_cols = np.where(~np.isnan(mb_data))
    if len(valid_rows) == 0:
        return geoms

    valid_xs, valid_ys = transform * (valid_cols + 0.5, valid_rows + 0.5)
    support_scores = []

    for geom in geoms:
        if geom is None or geom.is_empty:
            continue
        prepared_geom = prep(geom)
        score = 0

        for x, y in zip(valid_xs, valid_ys):
            if prepared_geom.contains(Point(float(x), float(y))):
                score += 1

        support_scores.append((score, geom))

    if not support_scores:
        return geoms

    supported = [geom for score, geom in support_scores if score > 0]
    if supported:
        return supported

    return [max(support_scores, key=lambda item: item[0])[1]]

for i, (g, ax) in enumerate(zip(glaciers_mb, axs.flat)):
    x0, y0, x1, y1 = g['bbox_map']

    ax.set_facecolor('white')

    gprp.imshow_grid(ax, g['mb_data'], g['tfm'],
                     cmap=cmap_mb, alpha=1.0, norm=norm_mb, zorder=5)

    if g['outline'] is not None:
        bh_pts = [Point(row.geometry.x, row.geometry.y)
                  for _, row in g['boreholes'].iterrows()]
        geoms = prepare_outline(g['outline'], g['bbox'], g['bbox_map'],
                                bh_points=bh_pts, bh_filter=g['bh_filter'])
        geoms = keep_supported_outline_geoms(explode_outline_geoms(geoms),
                                             g['mb_data'], g['tfm'])
        draw_outline(ax, geoms)

    # winter snow probe positions
    wx, wy = g['winter_pts']
    if len(wx):
        ax.scatter(wx, wy, s=18, marker='x', color='black',
                   linewidths=0.9, zorder=20)

    # annual ablation stake positions
    ax_x, ax_y = g['annual_pts']
    if len(ax_x):
        ax.scatter(ax_x, ax_y, s=35, marker='^', color='red',
                   edgecolors='black', linewidths=0.8, zorder=20)

    ax.text(0.03, 0.97, f'({chr(ord("a") + i)})', transform=ax.transAxes,
            ha='left', va='top', fontsize=ANNO_FONTSIZE, fontweight='bold', zorder=25,
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=1.0))

    valid   = g['mb_data'][~np.isnan(g['mb_data'])]
    ba_mean = float(np.mean(valid))
    ax.text(0.97, 0.97, f'$B_a$ = {ba_mean:.2f} m w.e.', transform=ax.transAxes,
            ha='right', va='top', fontsize=ANNO_FONTSIZE - 2, zorder=25,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.85))

    ax.text(0.03, 0.03, g['label'], transform=ax.transAxes,
            ha='left', va='bottom', fontsize=ANNO_FONTSIZE, fontweight='bold', zorder=25,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='black', alpha=0.85))

    ax.text(0.97, 0.03, '2024/25', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=ANNO_FONTSIZE - 2, zorder=25,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.85))

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect('equal', adjustable='box')
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.xaxis.set_major_formatter(fmt)
    ax.tick_params(axis='x', rotation=0, labelsize=8)
    ax.set_xlabel('Easting (LV95) [m]' if i // 3 == 1 else '', fontsize=ANNO_FONTSIZE)

    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.yaxis.set_major_formatter(fmt)
    ax.tick_params(axis='y', rotation=90, labelsize=8)
    ax.set_ylabel('Northing (LV95) [m]' if i % 3 == 0 else '', fontsize=ANNO_FONTSIZE)

# colorbar — left portion of bottom strip
bottom_shift = 0.03
cax_left = SP['left'] + bottom_shift
cax_w    = 0.44
cax = fig.add_axes([cax_left, 0.04, cax_w, 0.022])
cb  = fig.colorbar(plt.cm.ScalarMappable(norm=norm_mb, cmap=cmap_mb),
                   cax=cax, orientation='horizontal', extend='both')
cb.set_label('Annual mass balance $B_a$ [m w.e.]', fontsize=ANNO_FONTSIZE)
cb.ax.tick_params(labelsize=ANNO_FONTSIZE - 2)

# legend — right portion of bottom strip
legend_handles = [
    Line2D([0], [0], marker='x', color='black', linestyle='none',
           markersize=5, markeredgewidth=0.9, label='Winter snow probe'),
    Line2D([0], [0], marker='^', color='red', linestyle='none',
           markersize=5, markeredgewidth=0.8,
           markerfacecolor='red', markeredgecolor='black', label='Summer ablation stake'),
]
fig.legend(handles=legend_handles, loc='lower right', ncol=2,
           fontsize=ANNO_FONTSIZE - 2, framealpha=0.9, frameon=False,
           bbox_to_anchor=(0.92 + bottom_shift, 0.005))

out_path = output_dir + 'annual_balance_distribution.pdf'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()


---
## Figure 2 — Annual firn area time series (1970–2025)

Reads the `firn_area_annual_{glacier}.csv` files written by `firn_change.pro`
and plots the evolution of firn-covered area for all 6 glaciers.

In [ ]:
firn_dir = project_root + '/results/firn_grids/'

# glacier keys must match the names used in firn_change.pro
glaciers = [
    {'key': 'alphubel',  'label': 'Alphubel South',       'color': '#1f77b4'},
    {'key': 'felskinn',  'label': 'Chessjengletscher',     'color': '#ff7f0e'},
    {'key': 'hohsaas',   'label': 'Hohsaasgletscher',      'color': '#2ca02c'},
    {'key': 'sexrouge',  'label': 'Glacier du Sex Rouge',  'color': '#d62728'},
    {'key': 'tortin',    'label': 'Glacier de Tortin',     'color': '#9467bd'},
    {'key': 'corvatsch', 'label': 'Vadret dal Corvatsch',  'color': '#8c564b'},
]

CELL_AREA_KM2 = 10 * 10 * 1e-6  # 10 m cell

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=200)

for g in glaciers:
    csv_path = firn_dir + f'annual_area/firn_area_annual_{g["key"]}.csv'
    if not os.path.exists(csv_path):
        print(f'  NOT FOUND: {csv_path}')
        continue
    df = pd.read_csv(csv_path)
    df['firn_area_km2'] = df['firn_cells'] * CELL_AREA_KM2
    ax.plot(df['year'], df['firn_area_km2'],
            label=g['label'], color=g['color'], linewidth=1.5)

ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Firn area (km²)', fontsize=10)
ax.set_title('Annual firn-covered area 1970–2025', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, framealpha=0.9)
ax.grid(True, linewidth=0.4, alpha=0.5)
ax.set_xlim(1970, 2025)
ax.set_ylim(bottom=0)

fig.tight_layout()
out_path = output_dir + 'firn_area_timeseries.pdf'
fig.savefig(out_path, bbox_inches='tight', dpi=200)
print(f'Saved: {out_path}')
plt.show()

---
## Figure 3 — Annual firn area: individual panels

Same data as Figure 2 but one panel per glacier, showing also the glacier total area
as a reference (dashed line) so the firn fraction is visually apparent.

In [ ]:
# Total glacier areas from study sites table (km²)
glacier_areas = {
    'alphubel':  0.068,
    'felskinn':  0.205,
    'hohsaas':   0.151,
    'sexrouge':  0.222,
    'tortin':    0.495,
    'corvatsch': 0.143,
}

fig, axs = plt.subplots(2, 3, figsize=(12, 7), dpi=200,
                        sharex=True)
fig.subplots_adjust(left=0.07, right=0.98, top=0.93, bottom=0.10,
                    wspace=0.25, hspace=0.15)

for ax, g in zip(axs.flat, glaciers):
    csv_path = firn_dir + f'annual_area/firn_area_annual_{g["key"]}.csv'
    if not os.path.exists(csv_path):
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=9)
        ax.set_title(g['label'], fontsize=9, fontweight='bold')
        continue

    df = pd.read_csv(csv_path)
    df['firn_area_km2'] = df['firn_cells'] * CELL_AREA_KM2

    total_area = glacier_areas.get(g['key'])

    ax.fill_between(df['year'], df['firn_area_km2'],
                    alpha=0.35, color=g['color'])
    ax.plot(df['year'], df['firn_area_km2'],
            color=g['color'], linewidth=1.5, label='Firn area')
    if total_area is not None:
        ax.axhline(total_area, color='k', linewidth=1.0,
                   linestyle='--', label=f'Glacier area ({total_area:.3f} km²)')

    ax.set_title(g['label'], fontsize=9, fontweight='bold')
    ax.set_xlim(1970, 2025)
    ax.set_ylim(bottom=0)
    ax.grid(True, linewidth=0.4, alpha=0.5)
    ax.legend(fontsize=7, framealpha=0.9)

for ax in axs[1]:
    ax.set_xlabel('Year', fontsize=9)
for ax in axs[:, 0]:
    ax.set_ylabel('Firn area (km²)', fontsize=9)

out_path = output_dir + 'firn_area_timeseries_panels.pdf'
fig.savefig(out_path, bbox_inches='tight', dpi=200)
print(f'Saved: {out_path}')
plt.show()